# Alpamayo 1.5: Late-t0 Navigation Inference

This notebook uses the modified loader that supports partial future availability.

Behavior:
1. History must still be valid
2. Future may be partially missing at very late t0
3. If GT future exists, metrics and GT comparison are shown
4. If GT future does not exist, only predicted trajectories are shown
5. This allows using the very latest timestamps in the clip


In [ ]:
import os
import sys
import textwrap
from pathlib import Path

repo_root = Path.cwd().resolve().parent
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import av
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import BitsAndBytesConfig

from alpamayo1_5 import helper
from alpamayo1_5.load_offline_dataset import get_or_build_offline_clip_cache, load_offline_dataset
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.offline_eval_utils import (
    set_reproducible_seeds,
    run_offline_inference_window,
    _compute_adaptive_xy_limits,
)

print('repo_root =', repo_root)
print('src_path =', src_path)


In [ ]:
# ===== Reproducibility =====
SEED = 42
set_reproducible_seeds(SEED)

# ===== Paths =====
CLIP_DIR = Path('/workspace/dataset/')
MODEL_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Alpamayo-1.5-10B/snapshots/f11cd25b758ab560114019b555dde2a8b92d88b4')
PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b')
COSMOS_PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Cosmos-Reason2-8B/snapshots/f715d875a8a99a0a2b65aa28633afd9417e46bd9')

# ===== Camera =====
TARGET_CAMERA_FILENAME = 'Front_camera.mp4'
FRONT_FRAME_T = -1

# ===== Inference config =====
DEVICE = 'cuda'
NUM_HISTORY_STEPS = 16
NUM_FUTURE_STEPS = 64
TIME_STEP = 0.1
NUM_FRAMES = 4
FPS = 30.0
FRAME0_GPS_TIME_SOD = 175484.98
NUM_TRAJ_SAMPLES = 1
TEMPERATURE = 0.6
TOP_P = 0.98
MAX_GENERATION_LENGTH = 256
IMAGE_SIZE = (448, 800)
EVAL_XY_ROTATION_DEG = -90.0

# ===== Late t0 sweep =====
SWEEP_LAST_SECONDS = 5.0
SWEEP_STEP_SEC = 1.0

# ===== Prompts =====
NAV_TEXTS = [
    None,
    'Turn right in 20m',
    'Turn right in 10m',
    'Turn right in 5m',
    'Turn left in 10m',
]

PRINT_FULL_COT = True

os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'] = str(COSMOS_PROCESSOR_PATH)
print('NAV_TEXTS =', NAV_TEXTS)


In [ ]:
def pick_camera_row_by_exact_filename(clip_dir: Path, filename: str):
    camera_paths = helper.discover_offline_camera_files(clip_dir)
    camera_paths_sorted = sorted(camera_paths, key=lambda p: helper.infer_camera_index(p.name))
    matches = [(row, p) for row, p in enumerate(camera_paths_sorted) if p.name == filename]
    if len(matches) == 0:
        raise ValueError(f'No camera file matched exact filename={filename!r}')
    if len(matches) > 1:
        raise ValueError(f'Multiple camera files matched exact filename={filename!r}')
    return matches[0]


def decode_single_frame(video_path: Path, frame_index: int) -> np.ndarray:
    with av.open(str(video_path)) as container:
        stream = container.streams.video[0]
        for idx, frame in enumerate(container.decode(stream)):
            if idx == frame_index:
                return frame.to_ndarray(format='rgb24')
    raise IndexError(f'Frame index {frame_index} out of range for {video_path}')


def build_late_t0_list(cache):
    pose_min = float(cache.pose_times_sod[0])
    pose_max = float(cache.pose_times_sod[-1])
    history_left_span = (NUM_HISTORY_STEPS - 1) * TIME_STEP
    image_left_span = (NUM_FRAMES - 1) * TIME_STEP
    t0_min = max(pose_min + history_left_span, FRAME0_GPS_TIME_SOD + image_left_span)
    t0_end = pose_max
    t0_start = max(t0_min, t0_end - SWEEP_LAST_SECONDS)
    return list(np.arange(t0_start, t0_end + 1e-9, SWEEP_STEP_SEC, dtype=np.float64))


def render_window_figure(window_bundle):
    fig = plt.figure(figsize=(22, 8))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.4, 1.2])
    ax_img = fig.add_subplot(gs[0, 0])
    ax_bev = fig.add_subplot(gs[0, 1])
    ax_text = fig.add_subplot(gs[0, 2])

    ax_img.imshow(window_bundle['front_img'])
    ax_img.set_title(
        f"Front camera | frame={window_bundle['front_frame_idx']}\n"
        f"ts={window_bundle['front_frame_ts']:.3f}"
    )
    ax_img.axis('off')

    all_arrays = []
    first_key = list(window_bundle['results'].keys())[0]
    hist_xyz = window_bundle['results'][first_key]['hist_xyz_plot']
    all_arrays.append(hist_xyz)
    gt_available = False

    for nav_text, result in window_bundle['results'].items():
        all_arrays.append(result['pred_best_xyz'])
        if result['num_valid_future_steps'] >= 2 and result['gt_xyz_plot'] is not None and len(result['gt_xyz_plot']) > 0:
            all_arrays.append(result['gt_xyz_plot'])
            gt_available = True

    xlim, ylim = _compute_adaptive_xy_limits(*all_arrays, min_span=20.0, pad_ratio=0.12, pad_min=2.0)

    ax_bev.plot(hist_xyz[:, 0], hist_xyz[:, 1], 'b-o', linewidth=2.0, markersize=3.0, label='History')
    if gt_available:
        gt_xyz = window_bundle['results'][first_key]['gt_xyz_plot']
        if gt_xyz is not None and len(gt_xyz) > 0:
            ax_bev.plot(gt_xyz[:, 0], gt_xyz[:, 1], 'k-o', linewidth=3.0, markersize=3.2, label='GT(valid part)')

    color_map = {
        None: 'red',
        'Turn right in 20m': 'tab:blue',
        'Turn right in 10m': 'tab:orange',
        'Turn right in 5m': 'tab:green',
        'Turn left in 10m': 'tab:purple',
    }

    for nav_text, result in window_bundle['results'].items():
        label = 'no nav' if nav_text is None else nav_text
        pred = result['pred_best_xyz']
        ax_bev.plot(pred[:, 0], pred[:, 1], 'o-', color=color_map.get(nav_text, 'gray'), linewidth=2.2, markersize=3.2, label=label)

    ax_bev.scatter([0.0], [0.0], c='red', marker='x', s=120, label='t0 ego')
    ax_bev.set_xlabel('x')
    ax_bev.set_ylabel('y')
    ax_bev.set_title(f"Late-t0 navigation | t0={window_bundle['t0_sod']:.3f}")
    ax_bev.set_xlim(*xlim)
    ax_bev.set_ylim(*ylim)
    ax_bev.set_aspect('equal', adjustable='box')
    ax_bev.grid(True, alpha=0.3)
    ax_bev.legend(loc='best', fontsize=8)

    lines = [f"t0_sod: {window_bundle['t0_sod']:.3f}"]
    for nav_text, result in window_bundle['results'].items():
        label = 'no nav' if nav_text is None else nav_text
        m = result['df_metrics'].iloc[0].to_dict() if len(result['df_metrics']) else {}
        lines.append('')
        lines.append(label)
        lines.append(f"  valid_future_steps={result['num_valid_future_steps']}")
        lines.append(f"  minADE={m.get('minADE_m', np.nan)}")
        lines.append(f"  final_error={m.get('final_point_error_m', np.nan)}")
        lines.append(f"  cot={str(result.get('cot_value', ''))[:100]}")

    ax_text.text(
        0.01, 0.99,
        '\n'.join(lines),
        transform=ax_text.transAxes,
        va='top', ha='left', fontsize=8.5,
        bbox=dict(boxstyle='round', facecolor='whitesmoke', alpha=0.95),
    )
    ax_text.axis('off')
    ax_text.set_title('Metrics / CoT summary')

    plt.tight_layout()
    return fig


In [ ]:
clip_cache = get_or_build_offline_clip_cache(CLIP_DIR, debug=False, force_rebuild=False)
front_cam_row, front_video_path = pick_camera_row_by_exact_filename(CLIP_DIR, TARGET_CAMERA_FILENAME)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = Alpamayo1_5.from_pretrained(
    str(MODEL_PATH),
    dtype=torch.bfloat16,
    device_map='cuda:0',
    quantization_config=quantization_config,
)

processor = helper.get_processor(
    model.tokenizer,
    processor_name_or_path=PROCESSOR_PATH,
)

print('Loaded cache, model, processor, and front camera path.')
print('front_cam_row =', front_cam_row)
print('front_video_path =', front_video_path)


In [ ]:
t0_list = build_late_t0_list(clip_cache)
print('late t0 list =', t0_list)


In [ ]:
all_window_bundles = []

for i, t0_sod in enumerate(t0_list, start=1):
    print(f'\n==================== [{i}/{len(t0_list)}] t0_sod={t0_sod:.3f} ====================')
    data = load_offline_dataset(
        clip_dir=CLIP_DIR,
        t0_sod=float(t0_sod),
        num_history_steps=NUM_HISTORY_STEPS,
        num_future_steps=NUM_FUTURE_STEPS,
        time_step=TIME_STEP,
        num_frames=NUM_FRAMES,
        fps=FPS,
        frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
        debug=False,
        image_size=IMAGE_SIZE,
    )

    front_frame_idx = int(data['actual_video_frame_indices'][front_cam_row, FRONT_FRAME_T].item())
    front_frame_ts = float(data['absolute_timestamps_sod'][front_cam_row, FRONT_FRAME_T].item())
    front_img = decode_single_frame(front_video_path, front_frame_idx)

    results = {}
    for nav_text in NAV_TEXTS:
        result = run_offline_inference_window(
            data=data,
            model=model,
            processor=processor,
            device=DEVICE,
            top_p=TOP_P,
            temperature=TEMPERATURE,
            num_traj_samples=NUM_TRAJ_SAMPLES,
            max_generation_length=MAX_GENERATION_LENGTH,
            time_step=TIME_STEP,
            eval_xy_rotation_deg=EVAL_XY_ROTATION_DEG,
            nav_text=nav_text,
        )
        results[nav_text] = result

        label = 'no nav' if nav_text is None else nav_text
        m = result['df_metrics'].iloc[0].to_dict() if len(result['df_metrics']) else {}
        print(f"{label}: valid_future_steps={result['num_valid_future_steps']}, minADE={m.get('minADE_m', np.nan)}, final_error={m.get('final_point_error_m', np.nan)}")

    bundle = {
        't0_sod': float(t0_sod),
        'front_frame_idx': front_frame_idx,
        'front_frame_ts': front_frame_ts,
        'front_img': front_img,
        'results': results,
    }
    all_window_bundles.append(bundle)


In [ ]:
for bundle in all_window_bundles:
    fig = render_window_figure(bundle)
    plt.show()
    plt.close(fig)

    if PRINT_FULL_COT:
        print(f'\n----- t0={bundle["t0_sod"]:.3f} -----')
        for nav_text, result in bundle['results'].items():
            label = 'no nav' if nav_text is None else nav_text
            print(f'\n===== CoT ({label}) =====')
            print(result.get('cot_value', ''))
